In [2]:
!pip3 install pandas torch torchvision Pillow numpy matplotlib pathlib cairosvg


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Detect Device
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

Using device: mps


In [4]:
# Read CSV data
import csv
import pandas as pd

data = pd.read_csv('data/pokemon.csv')
print(f"Total number of samples: {len(data)}")
data.head()

Total number of samples: 1024


,id,name,primary_type,secondary_type,color,shape,sprite_file
0,1,bulbasaur,grass,poison,green,quadruped,sprites/0001.png
1,2,ivysaur,grass,poison,green,quadruped,sprites/0002.png
2,3,venusaur,grass,poison,green,quadruped,sprites/0003.png
3,4,charmander,fire,NaN,red,upright,sprites/0004.png
4,5,charmeleon,fire,NaN,red,upright,sprites/0005.png


In [5]:
# Build type, color and shape dictionaries
type_to_idx = {}
color_to_idx = {}
shape_to_idx = {}

for idx, type in enumerate(data['primary_type'].unique()):
    type_to_idx[type] = idx
for idx, color in enumerate(data['color'].unique()):
    color_to_idx[color] = idx
for idx, shape in enumerate(data['shape'].unique()):
    shape_to_idx[shape] = idx

print(type_to_idx)
print(color_to_idx)
print(shape_to_idx)


{'grass': 0, 'fire': 1, 'water': 2, 'bug': 3, 'normal': 4, 'poison': 5, 'electric': 6, 'ground': 7, 'fairy': 8, 'fighting': 9, 'psychic': 10, 'rock': 11, 'ghost': 12, 'ice': 13, 'dragon': 14, 'dark': 15, 'steel': 16, 'flying': 17}
{'green': 0, 'red': 1, 'blue': 2, 'white': 3, 'brown': 4, 'yellow': 5, 'purple': 6, 'pink': 7, 'gray': 8, 'black': 9}
{'quadruped': 0, 'upright': 1, 'armor': 2, 'squiggle': 3, 'bug-wings': 4, 'wings': 5, 'humanoid': 6, 'legs': 7, 'blob': 8, 'heads': 9, 'tentacles': 10, 'arms': 11, 'fish': 12, 'ball': 13}


In [23]:
import torchvision.transforms as transforms
from PIL import Image

class PokemonDataset(torch.utils.data.Dataset):
    def __init__(self, images, data, type_to_idx, color_to_idx, shape_to_idx):
        self.images = images
        self.data = data
        self.type_to_idx = type_to_idx
        self.color_to_idx = color_to_idx
        self.shape_to_idx = shape_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = f'data/{self.images[idx]}'
        image = Image.open(img_path).convert('RGB')
        transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
        image = transform(image)

        row = self.data.iloc[idx]
        
        type_idx = self.type_to_idx[row['primary_type']]
        color_idx = self.color_to_idx[row['color']]
        shape_idx = self.shape_to_idx[row['shape']]
        
        return (image, torch.tensor([type_idx, color_idx, shape_idx], dtype=torch.long))

In [24]:
dataset = PokemonDataset(data['sprite_file'], data, type_to_idx, color_to_idx, shape_to_idx)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Number of batches: {len(dataloader)}")
print(f"Batch size: {dataloader.batch_size}")
print(f"First batch shapes: {next(iter(dataloader))[0].shape}, {next(iter(dataloader))[1].shape}")

Number of batches: 16
Batch size: 64
First batch shapes: torch.Size([64, 3, 64, 64]), torch.Size([64, 3])


/Users/wayneyu/Documents/Kellogg/Spring 2026/MSAI 495/dexgen/.venv/lib/python3.13/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [25]:
import torch.nn as nn
class ConditionEmbedding(nn.Module):
    def __init__(self, num_types, num_colors, num_shapes, embedding_dims=[16, 8 ,8], condition_dim=32):
        super(ConditionEmbedding, self).__init__()
        self.type_embedding = nn.Embedding(num_types, embedding_dims[0])
        self.color_embedding = nn.Embedding(num_colors, embedding_dims[1])
        self.shape_embedding = nn.Embedding(num_shapes, embedding_dims[2])
        self.condition_embedding = nn.Linear(sum(embedding_dims), condition_dim)

    def forward(self, conditions):
        type_emb = self.type_embedding(conditions[:, 0])
        color_emb = self.color_embedding(conditions[:, 1])
        shape_emb = self.shape_embedding(conditions[:, 2])
        combined = torch.cat([type_emb, color_emb, shape_emb], dim=1)
        linear_emb = self.condition_embedding(combined)
        return torch.relu(linear_emb)

In [26]:
cond_emb = ConditionEmbedding(len(type_to_idx), len(color_to_idx), len(shape_to_idx)).to(device)
images, conditions = next(iter(dataloader))
out = cond_emb(conditions.to(device))
print(out.shape)

torch.Size([64, 32])


In [37]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, cond_dim):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.groupnorm1 = nn.GroupNorm(8, out_channels)
        self.activation = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.groupnorm2 = nn.GroupNorm(8, out_channels)
        self.cond_proj = nn.Linear(cond_dim, out_channels)
        self.skip_proj = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else None

    def forward(self, x, cond):
        residual = x
        out = self.conv1(x)
        out = self.groupnorm1(out)
        out = self.activation(out)
        cond_emb = self.cond_proj(cond).unsqueeze(2).unsqueeze(3)
        out = out + cond_emb
        out = self.conv2(out)
        out = self.groupnorm2(out)
        out = self.activation(out)
        if self.skip_proj:
            residual = self.skip_proj(residual)
        return out + residual

In [38]:
block = ResBlock(64, 128, cond_dim=32)
x = torch.randn(4, 64, 32, 32)
cond = torch.randn(4, 32)
print(block(x, cond).shape)  # expect [4, 128, 32, 32]

torch.Size([4, 128, 32, 32])
